# 08 — XGBoost

Este notebook inicia XGBoost mediante búsquedas pequeñas y secuenciales. Primero compara la base depurada de 24 variables con la poda de 17; luego ejecuta una grilla de cuatro configuraciones sobre el conjunto ganador. Si el mejor resultado cae en un extremo, se agregará una segunda grilla local antes de consultar el holdout temporal.

En esta primera ejecución 2017 permanece reservado. Esta separación evita lanzar una búsqueda amplia y evita utilizar el holdout para orientar hiperparámetros.

## Imports y resolución de la raíz

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)

In [ ]:
def find_project_root(start_path=None):
    """Busca la raíz del repositorio a partir del directorio actual."""
    start = Path(start_path or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").is_dir() and (candidate / "src").is_dir():
            return candidate
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encontró la raíz local del proyecto.")
project_root_str = str(PROJECT_ROOT)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)
print(f"Raíz del proyecto: {PROJECT_ROOT}")

In [ ]:
from src.config import RANDOM_STATE, RAW_DATA_DIR
from src.data import (
    combine_hotel_datasets,
    load_csv_with_fallback,
    make_temporal_holdout_split,
    normalize_text_values,
)
from src.evaluation import (
    cross_validate_model,
    get_grid_results,
    get_stratified_group_cross_validation,
    run_grid_search,
)
from src.features import (
    build_base_feature_set,
    build_tree_model_feature_sets,
    get_feature_types,
    make_exact_feature_groups,
)
from src.preprocessing import build_tabular_preprocessor

## Carga, conjuntos y desarrollo temporal

In [ ]:
H1_URL = "https://raw.githubusercontent.com/Lithium582/Obligatorio2026/refs/heads/main/data/raw/H1.csv"
H2_URL = "https://raw.githubusercontent.com/Lithium582/Obligatorio2026/refs/heads/main/data/raw/H2.csv"

h1_df = load_csv_with_fallback(RAW_DATA_DIR / "H1.csv", H1_URL)
h2_df = load_csv_with_fallback(RAW_DATA_DIR / "H2.csv", H2_URL)
bookings_df = normalize_text_values(combine_hotel_datasets(h1_df, h2_df))
X_reference, y = build_base_feature_set(bookings_df)
xgboost_feature_sets = build_tree_model_feature_sets(X_reference)

X_reference_train, X_reference_holdout, y_train, y_holdout = make_temporal_holdout_split(
    X_reference,
    y,
    period_values=X_reference["ArrivalDateYear"],
    validation_period=2017,
)
feature_sets_train = {
    name: frame.loc[X_reference_train.index].copy()
    for name, frame in xgboost_feature_sets.items()
}
feature_sets_holdout = {
    name: frame.loc[X_reference_holdout.index].copy()
    for name, frame in xgboost_feature_sets.items()
}
PRUNED_SET_NAME = "Tree-pruned | 17 features"
comparison_groups = make_exact_feature_groups(feature_sets_train[PRUNED_SET_NAME])

display(pd.DataFrame([
    {
        "feature_set": name,
        "feature_count": frame.shape[1],
        "development_rows": len(frame),
        "holdout_rows_reserved": len(feature_sets_holdout[name]),
    }
    for name, frame in feature_sets_train.items()
]))

## Pipeline y configuración fija

XGBoost utiliza `tree_method="hist"` para acelerar el entrenamiento. El paralelismo se administra desde la validación cruzada y cada estimador trabaja con un único hilo. `scale_pos_weight` se calcula exclusivamente sobre desarrollo.

In [ ]:
negative_count = y_train.eq(0).sum()
positive_count = y_train.eq(1).sum()
development_scale_pos_weight = negative_count / positive_count


def build_xgboost_pipeline(X, **model_parameters):
    numeric_features, categorical_features = get_feature_types(
        X, require_all_categorical=False
    )
    preprocessor = build_tabular_preprocessor(
        numeric_features=numeric_features,
        categorical_features=categorical_features,
        scale_numeric=False,
    )
    default_parameters = {
        "n_estimators": 150,
        "learning_rate": 0.08,
        "max_depth": 4,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
        "scale_pos_weight": development_scale_pos_weight,
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "n_jobs": 1,
        "random_state": RANDOM_STATE,
    }
    default_parameters.update(model_parameters)
    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", XGBClassifier(**default_parameters)),
    ])

print(f"scale_pos_weight de desarrollo: {development_scale_pos_weight:.4f}")

## Comparación de los conjuntos

Ambos conjuntos se evalúan con la misma configuración fija y los grupos calculados sobre la poda. Esta etapa requiere diez ajustes de 150 boosting rounds.

In [ ]:
feature_set_records = []
feature_set_fold_records = []
for feature_set_name, X_candidate in feature_sets_train.items():
    comparison_cv = get_stratified_group_cross_validation(n_splits=5)
    scores = cross_validate_model(
        estimator=build_xgboost_pipeline(X_candidate),
        X=X_candidate,
        y=y_train,
        scoring="roc_auc",
        cv=comparison_cv,
        groups=comparison_groups,
        n_jobs=-1,
    )
    feature_set_records.append({
        "feature_set": feature_set_name,
        "feature_count": X_candidate.shape[1],
        "cv_roc_auc_mean": scores.mean(),
        "cv_roc_auc_std": scores.std(),
    })
    feature_set_fold_records.extend([
        {"feature_set": feature_set_name, "fold": fold, "roc_auc": score}
        for fold, score in enumerate(scores, start=1)
    ])

feature_set_results = pd.DataFrame(feature_set_records).sort_values(
    "cv_roc_auc_mean", ascending=False
).reset_index(drop=True)
cleaned_base_score = feature_set_results.loc[
    feature_set_results["feature_set"].eq("Cleaned base | 24 features"),
    "cv_roc_auc_mean",
].iloc[0]
feature_set_results["delta_vs_cleaned_base"] = (
    feature_set_results["cv_roc_auc_mean"] - cleaned_base_score
)
feature_set_fold_results = pd.DataFrame(feature_set_fold_records)
selected_feature_set_name = feature_set_results.iloc[0]["feature_set"]
display(feature_set_results.round(6))
display(feature_set_fold_results.pivot(
    index="fold", columns="feature_set", values="roc_auc"
).round(6))
print(f"Conjunto seleccionado: {selected_feature_set_name}")

## Grilla 1 — complejidad local

La primera grilla varía únicamente profundidad y peso mínimo de los hijos. Son cuatro configuraciones, 20 ajustes y 150 boosting rounds por ajuste. `learning_rate`, muestreo y regularización permanecen fijos. Los resultados indicarán si hace falta desplazar alguno de los rangos.

In [ ]:
X_selected_train = feature_sets_train[selected_feature_set_name]
X_selected_holdout = feature_sets_holdout[selected_feature_set_name]
xgboost_stage_one_pipeline = build_xgboost_pipeline(X_selected_train)
xgboost_stage_one_param_grid = {
    "model__max_depth": [3, 5],
    "model__min_child_weight": [5, 15],
}
stage_one_grid_size = np.prod([len(values) for values in xgboost_stage_one_param_grid.values()])
print(
    f"Grilla 1: {stage_one_grid_size} configuraciones, "
    f"{stage_one_grid_size * 5} ajustes y 150 boosting rounds por ajuste."
)

In [ ]:
stage_one_cv = get_stratified_group_cross_validation(n_splits=5)
xgboost_stage_one_search = run_grid_search(
    estimator=xgboost_stage_one_pipeline,
    param_grid=xgboost_stage_one_param_grid,
    X=X_selected_train,
    y=y_train,
    scoring="roc_auc",
    cv=stage_one_cv,
    groups=comparison_groups,
    n_jobs=-1,
)
stage_one_results = get_grid_results(xgboost_stage_one_search)
stage_one_results["train_validation_gap"] = (
    stage_one_results["mean_train_score"] - stage_one_results["mean_test_score"]
)
stage_one_columns = [
    "params", "mean_train_score", "mean_test_score",
    "std_test_score", "train_validation_gap", "rank_test_score",
]
display(stage_one_results[stage_one_columns].round(6))
print(f"Mejores parámetros: {xgboost_stage_one_search.best_params_}")
print(f"Mejor ROC AUC CV: {xgboost_stage_one_search.best_score_:.6f}")

## Decisión pendiente

Después de ejecutar se revisará si `max_depth` o `min_child_weight` quedaron en un extremo y se observará el gap train–CV. Si corresponde, se agregará una segunda grilla pequeña alrededor de la región ganadora. El holdout temporal no se evalúa en esta versión del notebook.